# Combinações com escala pela média e desvio padrão

Nesta versão, usamos `(valor − média) / desvio padrão` em todas as medidas numéricas dos três modelos. A média e o desvio são calculados somente no treinamento de cada parte. O KNN mantém a imputação pela mediana; a árvore, pela média; a rede mantém a preparação que já usava. A escala é a mesma fórmula, com o tratamento de ausências declarado para cada componente.

O KNN continua com K=11 e as demais decisões internas da versão anterior. A árvore continua com quatro medidas e a rede mantém quatro entradas, 12 neurônios ocultos e 800 épocas. A busca de misturas é repetida para esta nova preparação; os resultados ficam em arquivos próprios.

### Resultado com a nova escala

Adotamos `(valor − média) / desvio padrão` para as medidas numéricas de todos os modelos desta versão. Os parâmetros são ajustados somente no treinamento de cada parte.

| Modelo ou combinação | F1 macro na reserva de 104 amostras |
|---|---:|
| KNN K=11 com média e desvio padrão | 75,94% |
| Árvore com média e desvio padrão | 84,61% |
| Rede neural, que já usava essa fórmula | 85,58% |
| Trio escolhido nesta versão | 86,49% |

Repetimos a grade de 79.326 misturas para a nova preparação e levamos 39 candidatas à segunda seleção. O trio escolhido usa **32,5% KNN, 42,5% árvore e 25% rede**, com limite **45,5%**. O F1 macro médio nas 30 partes de seleção foi **91,53%**. Na reserva acertou **90 de 104 amostras**, com acurácia 86,54% e matriz `[[42, 8], [6, 48]]`.

A mudança de escala reduziu o resultado do KNN nessa reserva: a versão anterior tinha F1 de 86,53%. O trio anterior, que usava quartis no KNN, tinha 88,44%. Esses resultados anteriores ficam como histórico; a versão atual segue a fórmula escolhida pelo aluno.

O arquivo desta mistura é `resultados/submissao_combinacao_zscore.csv`. O KNN individual está em `resultados/submissao_knn_zscore.csv`. As novas previsões ainda não foram enviadas ao Kaggle nesta execução.

### Como vamos comparar

Usamos as mesmas 416 amostras de treinamento e 104 reservadas dos outros notebooks. A base tem um registro por paciente. Cada modelo aprende o preenchimento e a normalização somente na parte usada para treinar em cada divisão.

Fazemos uma busca ampla em cinco partes das sementes 42, 73 e 101. Depois comparamos uma lista menor nas sementes 21, 145 e 307, para escolher a regra pelo F1 macro médio das seis divisões. Os empates favorecem menos componentes e uma regra mais simples.

Fixamos a escolha antes das conferências nas sementes 509 e 911 e antes das conferências com outras inicializações da rede. Só depois observamos a comparação nas 104 amostras reservadas. Essas pessoas já apareceram em outros testes do projeto; mesmo as novas divisões não equivalem a um conjunto externo novo.

As pontuações públicas do Kaggle ficam como histórico: KNN 0,82897; árvore 0,83018; mistura original 0,84382. Elas não entram no cálculo que escolhe a nova regra.

In [1]:
from pathlib import Path
from statistics import median, quantiles
from time import perf_counter
import csv
import hashlib
import json
import math
import random

import numpy as np

raiz = Path.cwd()
if not (raiz / "dados" / "treinamento_padronizado.csv").is_file():
    raiz = raiz.parent
caminho_base = raiz / "dados" / "treinamento_padronizado.csv"
with caminho_base.open(encoding="utf-8-sig", newline="") as arquivo:
    amostras = list(csv.DictReader(arquivo))

numericas = (
    "AGE", "HEIGHT", "WEIGHT", "SYSTOLIC", "DIASTOLIC", "GLUCOSE",
    "HEMOGLOBIN", "TRIGLYCERIDES", "SODIUM",
)
indices_arvore = tuple(numericas.index(c) for c in ("SYSTOLIC", "GLUCOSE", "HEMOGLOBIN", "TRIGLYCERIDES"))
classes = np.array([int(a["class"]) for a in amostras], dtype=np.int8)
pacientes = [a["PATIENT_CODE"].strip() for a in amostras]
if any(not p for p in pacientes) or len(set(pacientes)) != len(pacientes):
    raise ValueError("Conferir os registros de paciente.")
if not np.all(np.isin(classes, [0, 1])):
    raise ValueError("Esperávamos classes 0 e 1.")

controle = np.flatnonzero(classes == 0).tolist()
diabetes = np.flatnonzero(classes == 1).tolist()
gerador = random.Random(42)
gerador.shuffle(controle)
gerador.shuffle(diabetes)
n0, n1 = round(len(controle) * 0.20), round(len(diabetes) * 0.20)
reservadas = np.array(controle[:n0] + diabetes[:n1])
treinamento = np.array(controle[n0:] + diabetes[n1:])


def dividir(indices, semente):
    partes = [[] for _ in range(5)]
    gerador = random.Random(semente)
    for classe in (0, 1):
        grupo = [int(i) for i in indices if classes[i] == classe]
        gerador.shuffle(grupo)
        for posicao, i in enumerate(grupo):
            partes[posicao % 5].append(i)
    for posicao, parte in enumerate(partes):
        referencia = [i for j, outra in enumerate(partes) if j != posicao for i in outra]
        yield np.array(referencia), np.array(parte)


def ler_numero(amostra, coluna):
    try:
        valor = float(str(amostra[coluna]).strip().replace(",", "."))
    except ValueError:
        return None
    if not math.isfinite(valor) or valor <= 0:
        return None
    if coluna == "HEMOGLOBIN" and valor > 100:
        return None
    return valor


print(f"Treinamento: {len(treinamento)} | Reserva: {len(reservadas)}")
print("SHA-256 da base:", hashlib.sha256(caminho_base.read_bytes()).hexdigest())

from itertools import combinations, product
import sys

print(f"Python {sys.version.split()[0]} | NumPy {np.__version__}")

Treinamento: 416 | Reserva: 104
SHA-256 da base: bc1a686d83c06e2d7667a6089e82b54b4bff2ca6467b81155f1246ac4719a032
Python 3.12.14 | NumPy 2.3.5


### Resposta do KNN

O apoio é a fração dos 11 vizinhos de diabetes. As medidas numéricas agora usam média e desvio padrão, calculados nos valores disponíveis do treinamento. Os vazios recebem a mediana. Os demais componentes da distância são os mesmos da experiência anterior: pesos internos, Manhattan e tratamento do tabagismo.

In [2]:
pesos_knn = [2 if c in {"GLUCOSE", "HEMOGLOBIN"} else 1 for c in numericas]


def fumante(amostra):
    valor = amostra["SMOKER"].strip()
    return "" if valor in {"", "desconhecido"} else valor


def estimar_escala(referencia, colunas):
    parametros = {}
    for coluna in colunas:
        valores = [ler_numero(a, coluna) for a in referencia]
        disponiveis = np.array([v for v in valores if v is not None], dtype=float)
        if len(disponiveis) < 2:
            raise ValueError(f"Valores insuficientes para {coluna}.")
        parametros[coluna] = {
            "media": float(np.mean(disponiveis)),
            "desvio": float(np.std(disponiveis, ddof=1)),
            "mediana": float(np.median(disponiveis)),
        }
    return parametros


def preparar_knn(referencia):
    return estimar_escala(referencia, numericas)


def normalizar_knn(amostra, parametros):
    resultado = []
    for coluna in numericas:
        regra = parametros[coluna]
        valor = ler_numero(amostra, coluna)
        if valor is None:
            valor = regra["mediana"]
        resultado.append((valor - regra["media"]) / regra["desvio"] if regra["desvio"] else 0.0)
    return resultado


def apoiar_knn(referencia, consulta):
    parametros = preparar_knn(referencia)
    categorias_conhecidas = {fumante(a) for a in referencia if fumante(a)}
    vizinhos_referencia = [
        (normalizar_knn(a, parametros), fumante(a), int(a["class"]))
        for a in referencia
    ]
    random.Random(42).shuffle(vizinhos_referencia)
    apoios = []

    for amostra in consulta:
        numeros = normalizar_knn(amostra, parametros)
        categoria = fumante(amostra)
        vizinhos = []

        for numeros_ref, categoria_ref, classe in vizinhos_referencia:
            soma = 0.0
            for j, peso in enumerate(pesos_knn):
                soma += peso * abs(numeros[j] - numeros_ref[j])
            media = soma / sum(pesos_knn)
            distancia = media / (1 + media)
            if categoria and categoria_ref and categoria in categorias_conhecidas:
                jaccard = 0.0 if categoria == categoria_ref else 1.0
                distancia = 0.90 * distancia + 0.10 * jaccard
            vizinhos.append((distancia, classe))

        vizinhos.sort(key=lambda vizinho: vizinho[0])
        votos_diabetes = sum(classe for distancia, classe in vizinhos[:11])
        apoios.append(votos_diabetes / 11)

    apoios = np.array(apoios)
    return apoios, (apoios > 0.5).astype(np.int8)

### Resposta da árvore

A árvore continua com mínimo de 15 exemplos por folha e profundidade máxima 6, após a simplificação de ramos. A classe individual é a classe que a árvore escolhe na folha alcançada.

Para as médias, o apoio é a proporção de exemplos de diabetes nessa folha. Essa proporção e a saída da rede não foram calibradas como probabilidades. As regras que usam seus valores como grau de apoio são hipóteses que vamos comparar.

In [3]:
def entropia(positivos, total):
    proporcao = np.asarray(positivos, dtype=float) / total
    p0 = 1 - proporcao
    parte1 = proporcao * np.log2(np.maximum(proporcao, 1e-15))
    parte0 = p0 * np.log2(np.maximum(p0, 1e-15))
    return -(parte1 + parte0)


def melhor_teste(x, y, atributos, minimo_folha):
    total = len(y)
    positivos = int(y.sum())
    antes = float(entropia(positivos, total))
    melhor = None
    melhor_ganho = 1e-12

    for j in atributos:
        valores = x[:, j]
        if j < len(numericas):
            ordem = np.argsort(valores, kind="stable")
            ordenados = valores[ordem]
            quantidades = np.arange(1, total)
            positivos_esquerda = np.cumsum(y[ordem])[:-1]
            candidatos = (
                (ordenados[:-1] < ordenados[1:])
                & (quantidades >= minimo_folha)
                & (total - quantidades >= minimo_folha)
            )
            if not candidatos.any():
                continue
            posicoes = np.flatnonzero(candidatos)
            n_esq = quantidades[posicoes]
            p_esq = positivos_esquerda[posicoes]
            depois = (
                n_esq * entropia(p_esq, n_esq)
                + (total - n_esq) * entropia(positivos - p_esq, total - n_esq)
            ) / total
            ganhos = antes - depois
            k = int(np.argmax(ganhos))
            ganho = float(ganhos[k])
            posicao = int(posicoes[k])
            limite = float((ordenados[posicao] + ordenados[posicao + 1]) / 2)
            tipo = "numero"
        else:
            unicos = np.unique(valores)
            ganhos = []
            limites = []
            for valor in unicos:
                esquerda = valores == valor
                n_esq = int(esquerda.sum())
                if n_esq < minimo_folha or total - n_esq < minimo_folha:
                    continue
                p_esq = int(y[esquerda].sum())
                depois = (
                    n_esq * entropia(p_esq, n_esq)
                    + (total - n_esq) * entropia(positivos - p_esq, total - n_esq)
                ) / total
                ganhos.append(float(antes - depois))
                limites.append(float(valor))
            if not ganhos:
                continue
            k = int(np.argmax(ganhos))
            ganho, limite = ganhos[k], limites[k]
            tipo = "categoria"

        if ganho > melhor_ganho:
            melhor_ganho = ganho
            melhor = (j, tipo, limite, ganho)
    return melhor

In [4]:
def construir_arvore(x, y, atributos, minimo_folha, padrao=0):
    positivos = int(y.sum())
    if positivos * 2 == len(y):
        classe = padrao
    else:
        classe = int(positivos * 2 > len(y))
    no = {"classe": classe, "amostras": len(y), "positivos": positivos}
    if positivos == 0 or positivos == len(y) or len(y) < 2 * minimo_folha:
        return no

    teste = melhor_teste(x, y, atributos, minimo_folha)
    if teste is None:
        return no
    j, tipo, limite, ganho = teste
    esquerda = x[:, j] <= limite if tipo == "numero" else x[:, j] == limite
    no.update(atributo=j, tipo=tipo, limite=limite, ganho=ganho)
    if tipo == "categoria":
        no["conhecidas"] = np.unique(x[:, j]).tolist()
    no["sim"] = construir_arvore(x[esquerda], y[esquerda], atributos, minimo_folha, classe)
    no["nao"] = construir_arvore(x[~esquerda], y[~esquerda], atributos, minimo_folha, classe)
    return no


def simplificar_arvore(no, profundidade, nivel=0):
    folha = {
        "classe": no["classe"],
        "amostras": no["amostras"],
        "positivos": no["positivos"],
    }
    if "atributo" not in no or (profundidade is not None and nivel >= profundidade):
        return folha

    copia = no.copy()
    copia["sim"] = simplificar_arvore(no["sim"], profundidade, nivel + 1)
    copia["nao"] = simplificar_arvore(no["nao"], profundidade, nivel + 1)
    sim, nao = copia["sim"], copia["nao"]

    if "atributo" not in sim and "atributo" not in nao and sim["classe"] == nao["classe"]:
        if no["tipo"] == "numero" or sim["classe"] == no["classe"]:
            folha["classe"] = sim["classe"]
            return folha
    return copia

In [5]:
def matriz_arvore(amostras_entrada, medias=None):
    numeros = np.array([
        [ler_numero(a, c) for c in numericas] for a in amostras_entrada
    ], dtype=float)
    if medias is None:
        medias = []
        for j, coluna in enumerate(numericas):
            disponiveis = numeros[np.isfinite(numeros[:, j]), j]
            if not len(disponiveis):
                raise ValueError(f"Nenhuma medida disponível em {coluna}.")
            medias.append(float(np.mean(disponiveis)))
    for j, valor in enumerate(medias):
        numeros[:, j] = np.where(np.isfinite(numeros[:, j]), numeros[:, j], valor)
    return numeros, medias


def folha_alcancada(entrada, arvore):
    no = arvore
    while "atributo" in no:
        caminho = "sim" if entrada[no["atributo"]] <= no["limite"] else "nao"
        no = no[caminho]
    return no


def apoiar_arvore(referencia, consulta):
    parametros = estimar_escala(referencia, numericas)
    x, medias = matriz_arvore(referencia)
    aval, _ = matriz_arvore(consulta, medias)
    centros = np.array([parametros[c]["media"] for c in numericas])
    desvios = np.array([parametros[c]["desvio"] for c in numericas])
    x = np.divide(x - centros, desvios, out=np.zeros_like(x), where=desvios != 0)
    aval = np.divide(aval - centros, desvios, out=np.zeros_like(aval), where=desvios != 0)
    y = np.array([int(a["class"]) for a in referencia], dtype=np.int8)
    arvore = construir_arvore(x, y, indices_arvore, 15)
    arvore = simplificar_arvore(arvore, 6)
    folhas = [folha_alcancada(a, arvore) for a in aval]
    apoios = np.array([no["positivos"] / no["amostras"] for no in folhas])
    previstas = np.array([no["classe"] for no in folhas], dtype=np.int8)
    return apoios, previstas


def combinar(apoio_knn, apoio_arvore, peso_arvore):
    apoio = (1 - peso_arvore) * apoio_knn + peso_arvore * apoio_arvore
    previstas = (apoio > 0.5).astype(np.int8)
    empate = np.isclose(apoio, 0.5, rtol=0, atol=1e-12)
    previstas[empate] = (apoio_knn[empate] > 0.5).astype(np.int8)
    return previstas


def contar(reais, previstas):
    return np.array([
        np.sum((reais == 0) & (previstas == 0)),
        np.sum((reais == 0) & (previstas == 1)),
        np.sum((reais == 1) & (previstas == 0)),
        np.sum((reais == 1) & (previstas == 1)),
    ])


def razao(a, b):
    return np.divide(a, b, out=np.zeros_like(a, dtype=float), where=b != 0)


def metricas(contagens):
    tn, fp, fn, tp = np.moveaxis(np.asarray(contagens), -1, 0)
    return {
        "acuracia": razao(tn + tp, tn + fp + fn + tp),
        "f1": (razao(2 * tn, 2 * tn + fp + fn) + razao(2 * tp, 2 * tp + fp + fn)) / 2,
        "precisao": (razao(tn, tn + fn) + razao(tp, tp + fp)) / 2,
        "recall0": razao(tn, tn + fp),
        "recall1": razao(tp, tp + fn),
    }


def mostrar(nome, contagens):
    m = metricas(contagens)
    print(f"{nome:24} | " + " | ".join(
        f"{np.mean(m[c]):>8.2%}" for c in ("acuracia", "f1", "precisao", "recall0", "recall1")
    ))

### Resposta da rede neural

Mantemos mediana para os vazios, média e desvio padrão para normalizar, quatro entradas, 12 neurônios ocultos e uma saída sigmoide. A saída maior que 0,5 indica diabetes. A inicialização e o embaralhamento usam a semente 7, exceto nas conferências explicitamente indicadas.

O treinamento é o mesmo do notebook da rede: retropropagação do erro quadrático, com atualização depois de cada exemplo. Usamos o nome `ler_numero_rn` porque a rede representa ausências com `NaN`, enquanto o primeiro KNN usa `None`.

In [6]:
grupos_rn = {"Quatro medidas": ("SYSTOLIC", "GLUCOSE", "HEMOGLOBIN", "TRIGLYCERIDES")}

def ler_numero_rn(amostra, coluna):
    try:
        valor = float(str(amostra[coluna]).strip().replace(",", "."))
    except ValueError:
        return np.nan
    if not np.isfinite(valor) or valor <= 0:
        return np.nan
    if coluna == "HEMOGLOBIN" and valor > 100:
        return np.nan
    return valor


def ajustar_preparo(referencia, grupo, escala, preenchimento):
    parametros = {"grupo": grupo, "escala": escala, "preenchimento": preenchimento, "numeros": {}}
    for coluna in grupos_rn[grupo]:
        valores = np.array([ler_numero_rn(a, coluna) for a in referencia])
        disponiveis = valores[np.isfinite(valores)]
        if not len(disponiveis):
            raise ValueError(f"Nenhuma medida disponível para {coluna}.")
        estimativa = np.mean(disponiveis) if preenchimento == "media" else np.median(disponiveis)
        if escala == "minmax":
            centro = np.min(disponiveis)
            divisor = np.ptp(disponiveis)
        else:
            centro = np.mean(disponiveis)
            divisor = np.std(disponiveis, ddof=1) if len(disponiveis) > 1 else 0
        parametros["numeros"][coluna] = {
            "estimativa": float(estimativa), "centro": float(centro), "divisor": float(divisor),
        }
    parametros["entradas"] = list(grupos_rn[grupo])
    return parametros


def transformar(consulta, parametros):
    linhas = []
    for amostra in consulta:
        linha = []
        for coluna in grupos_rn[parametros["grupo"]]:
            regra = parametros["numeros"][coluna]
            valor = ler_numero_rn(amostra, coluna)
            if not np.isfinite(valor):
                valor = regra["estimativa"]
            normalizado = (valor - regra["centro"]) / regra["divisor"] if regra["divisor"] else 0.0
            linha.append(normalizado)
        linhas.append(linha)
    return np.array(linhas, dtype=float)

In [7]:
def sigmoide(z):
    resultado = np.empty_like(z, dtype=float)
    positivos = z >= 0
    resultado[positivos] = 1 / (1 + np.exp(-z[positivos]))
    exponencial = np.exp(z[~positivos])
    resultado[~positivos] = exponencial / (1 + exponencial)
    return resultado


def incluir_vies(x):
    return np.column_stack((x, -np.ones(len(x))))


def passar(x_com_vies, pesos_entrada, pesos_saida):
    oculta = sigmoide(x_com_vies @ pesos_entrada)
    oculta_com_vies = incluir_vies(oculta)
    saida = sigmoide(oculta_com_vies @ pesos_saida)
    return oculta, oculta_com_vies, saida


def prever(x, modelo):
    _, _, saida = passar(incluir_vies(x), modelo[0], modelo[1])
    return (saida.ravel() > 0.5).astype(np.int8)

def treinar_exemplos(x, y, ocultos, taxa, pontos, semente=7, registrar=False):
    gerador = np.random.default_rng(semente)
    pesos_entrada = gerador.uniform(-0.5, 0.5, size=(x.shape[1] + 1, ocultos))
    pesos_saida = gerador.uniform(-0.5, 0.5, size=ocultos + 1)
    x_com_vies = incluir_vies(x)
    y = np.asarray(y, dtype=float)
    oculta_com_vies = np.empty(ocultos + 1)
    oculta_com_vies[-1] = -1
    modelos = {}
    historico = []

    with np.errstate(over="ignore"):
        for epoca in range(1, max(pontos) + 1):
            for i in gerador.permutation(len(x)):
                oculta = 1 / (1 + np.exp(-(x_com_vies[i] @ pesos_entrada)))
                oculta_com_vies[:-1] = oculta
                saida = 1 / (1 + np.exp(-(oculta_com_vies @ pesos_saida)))
                delta_saida = (y[i] - saida) * saida * (1 - saida)
                delta_oculta = delta_saida * pesos_saida[:-1] * oculta * (1 - oculta)
                pesos_entrada += taxa * np.outer(x_com_vies[i], delta_oculta)
                pesos_saida += taxa * oculta_com_vies * delta_saida

            if epoca in pontos or (registrar and epoca % 20 == 0):
                modelo = (pesos_entrada.copy(), pesos_saida.reshape(-1, 1).copy())
                _, _, respostas = passar(x_com_vies, *modelo)
                erro = float(np.mean((y - respostas.ravel()) ** 2) / 2)
                if not np.isfinite(erro) or not all(np.isfinite(p).all() for p in modelo):
                    raise ValueError("O treinamento produziu pesos ou erro inválidos.")
                if registrar:
                    historico.append((epoca, erro))
                if epoca in pontos:
                    modelos[epoca] = modelo
    return modelos, historico

def apoiar_rede(referencia, consulta, semente=7):
    preparo = ajustar_preparo(referencia, "Quatro medidas", "desvio", "mediana")
    x = transformar(referencia, preparo)
    aval = transformar(consulta, preparo)
    y = np.array([int(a["class"]) for a in referencia])
    modelos, _ = treinar_exemplos(x, y, 12, 0.1, (800,), semente)
    modelo = modelos[800]
    _, _, saida = passar(incluir_vies(aval), *modelo)
    apoio = saida.ravel()
    return apoio, (apoio > 0.5).astype(np.int8)


def prever_tres(referencia, consulta, semente_rede=7):
    resultados = [
        apoiar_knn(referencia, consulta),
        apoiar_arvore(referencia, consulta),
        apoiar_rede(referencia, consulta, semente_rede),
    ]
    apoios = np.column_stack([r[0] for r in resultados])
    rotulos = np.column_stack([r[1] for r in resultados])
    if not np.isfinite(apoios).all() or np.any((apoios < 0) | (apoios > 1)):
        raise ValueError("Apoios inválidos.")
    return apoios, rotulos

### Conferência dos componentes e registro das previsões

A árvore recebe os valores na nova escala. Como a transformação preserva a ordem, conferimos se suas previsões continuam equivalentes às anteriores. A rede já usava essa escala e deve reproduzir seu CSV. O KNN pode mudar com a nova escala; registramos seu novo CSV e repetimos a avaliação das misturas.

Podemos reaproveitar as respostas da rede da avaliação anterior, que usou os mesmos dados, preparação, configuração, sementes e partes. Conferimos a identificação da base, os índices e as previsões no teste. Se esse registro não estiver disponível, treinamos a rede normalmente.

In [8]:
def padronizar_teste(amostra):
    entrada = amostra.copy()
    if entrada["AGE"].strip() in {"0", "-1", "999"}:
        entrada["AGE"] = ""
    altura = ler_numero(entrada, "HEIGHT")
    if altura is not None and altura < 3:
        entrada["HEIGHT"] = str(altura * 100)
    pressao = entrada["BLOOD_PRESSURE"].strip().replace("x", "/").replace(" ", "")
    if pressao in {"", "?"}:
        entrada["SYSTOLIC"], entrada["DIASTOLIC"] = "", ""
    else:
        sistolica, diastolica = pressao.split("/")
        entrada["SYSTOLIC"] = str(int(sistolica))
        entrada["DIASTOLIC"] = str(int(diastolica))
    mapa = {
        "never": "nunca_fumou", "former": "ex_fumante", "ex-smoker": "ex_fumante",
        "current": "fumante_atual", "yes": "fumante_atual", "": "",
    }
    valor = entrada["SMOKER"].strip().lower()
    entrada["SMOKER"] = mapa.get(valor, valor)
    return entrada


with (raiz / "dados" / "kaggle-test" / "test.csv").open(
    encoding="utf-8-sig", newline=""
) as arquivo:
    teste_original = list(csv.DictReader(arquivo))
ids_teste = [a["id"] for a in teste_original]
if not ids_teste or len(set(ids_teste)) != len(ids_teste) or any(not i.strip() for i in ids_teste):
    raise ValueError("O teste precisa ter IDs únicos e preenchidos.")
teste = [padronizar_teste(a) for a in teste_original]


versao_modelos = "knn11-zscore_arvore4-zscore-f15-d6_rn4-12-1-t01-e800-v1"
sha_base = hashlib.sha256(caminho_base.read_bytes()).hexdigest()
caminho_cache = raiz / "resultados" / "combinacoes_zscore_cache.npz"
cache = {}
if caminho_cache.exists():
    with np.load(caminho_cache, allow_pickle=False) as anterior:
        if anterior["sha_base"].item() != sha_base or anterior["versao"].item() != versao_modelos:
            raise ValueError("O registro anterior pertence a outra base ou versão dos modelos.")
        cache = {k: anterior[k].copy() for k in anterior.files}


def salvar_cache():
    np.savez_compressed(caminho_cache, **cache)


if "apoios_teste" not in cache:
    apoios, rotulos = prever_tres(amostras, teste)
    cache.update(sha_base=np.array(sha_base), versao=np.array(versao_modelos),
                 apoios_teste=apoios, rotulos_teste=rotulos,
                 ids_teste=np.array(ids_teste), indices_treinamento=treinamento, indices_reserva=reservadas)
    salvar_cache()

apoios_teste, rotulos_teste = cache["apoios_teste"], cache["rotulos_teste"]
if cache["ids_teste"].tolist() != ids_teste:
    raise ValueError("IDs do teste diferentes do registro.")


def conferir_original(nome, previstas):
    with (raiz / "resultados" / nome).open(encoding="utf-8-sig", newline="") as arquivo:
        leitor = csv.DictReader(arquivo)
        if leitor.fieldnames != ["id", "class"]:
            raise ValueError("Cabeçalho inesperado.")
        linhas = list(leitor)
    if [a["id"] for a in linhas] != ids_teste or [int(a["class"]) for a in linhas] != previstas.tolist():
        raise ValueError(f"Não reproduzimos {nome}.")


for j, nome in ((1, "submissao_arvore_4_atributos.csv"), (2, "submissao_rede_neural_exemplos.csv")):
    conferir_original(nome, rotulos_teste[:, j])
print("Árvore e rede reproduzem as submissões anteriores.")
with (raiz / "resultados" / "submissao_knn_zscore.csv").open("w", encoding="utf-8", newline="") as arquivo:
    escritor = csv.writer(arquivo)
    escritor.writerow(["id", "class"])
    escritor.writerows(zip(ids_teste, rotulos_teste[:, 0].tolist(), strict=True))

historico_rede = {}
caminho_historico = raiz / "resultados" / "combinacoes_tres_cache.npz"
if caminho_historico.exists():
    with np.load(caminho_historico, allow_pickle=False) as antigo:
        valido = (antigo["sha_base"].item() == sha_base
                  and np.array_equal(antigo["indices_treinamento"], treinamento)
                  and np.array_equal(antigo["indices_reserva"], reservadas)
                  and antigo["ids_teste"].tolist() == ids_teste
                  and "rn4-12-1-t01-e800" in antigo["versao"].item()
                  and np.allclose(antigo["apoios_teste"][:, 2], apoios_teste[:, 2], rtol=0, atol=1e-12))
        if valido:
            historico_rede = {k: antigo[k].copy() for k in antigo.files}
print("Registro da rede disponível para reaproveitar:", bool(historico_rede))

Árvore e rede reproduzem as submissões anteriores.
Registro da rede disponível para reaproveitar: True


#### Respostas fora do treinamento de cada parte

Em cada divisão, cada amostra recebe uma previsão feita por modelos que não treinaram com ela. As três respostas ficam alinhadas na mesma ordem, junto com o número da parte. As conferências podem ser calculadas agora, mas suas métricas só serão observadas depois de fixar a escolha.

In [9]:
sementes_busca = (42, 73, 101)
sementes_refinamento = (21, 145, 307)
sementes_conferencia = (509, 911)
condicoes = [(s, 7) for s in sementes_busca + sementes_refinamento + sementes_conferencia]
condicoes += [(42, 19), (42, 31)]
posicao_treino = {int(i): j for j, i in enumerate(treinamento)}


def obter_divisao(semente_divisao, semente_rede=7):
    chave = f"{semente_divisao}_{semente_rede}"
    if "apoios_" + chave not in cache:
        inicio = perf_counter()
        apoios = np.full((len(treinamento), 3), np.nan)
        rotulos = np.full((len(treinamento), 3), -1, dtype=np.int8)
        partes = np.full(len(treinamento), -1, dtype=np.int8)
        for numero, (ref, aval) in enumerate(dividir(treinamento, semente_divisao)):
            if set(ref) & set(aval) or (set(ref) | set(aval)) & set(reservadas):
                raise ValueError("Sobreposição entre treinamento e avaliação.")
            referencia = [amostras[i] for i in ref]
            consulta = [amostras[i] for i in aval]
            posicoes = [posicao_treino[int(i)] for i in aval]
            if "apoios_" + chave in historico_rede:
                ak, rk = apoiar_knn(referencia, consulta)
                aa, ra = apoiar_arvore(referencia, consulta)
                an = historico_rede["apoios_" + chave][posicoes, 2]
                rn = historico_rede["rotulos_" + chave][posicoes, 2]
                a, r = np.column_stack((ak, aa, an)), np.column_stack((rk, ra, rn))
            else:
                a, r = prever_tres(referencia, consulta, semente_rede)
            if np.any(partes[posicoes] != -1):
                raise ValueError("Amostra avaliada duas vezes na mesma divisão.")
            apoios[posicoes], rotulos[posicoes], partes[posicoes] = a, r, numero
        if not np.isfinite(apoios).all() or np.any(partes < 0):
            raise ValueError("Faltam previsões na divisão.")
        cache["apoios_" + chave] = apoios
        cache["rotulos_" + chave] = rotulos
        cache["partes_" + chave] = partes
        salvar_cache()
        print(f"Divisão {semente_divisao}, rede {semente_rede}: {perf_counter() - inicio:.1f} s", flush=True)
    return cache["apoios_" + chave], cache["rotulos_" + chave], cache["partes_" + chave]


for divisao, semente_rede in condicoes:
    obter_divisao(divisao, semente_rede)

if "apoios_reserva" not in cache:
    a, r = prever_tres([amostras[i] for i in treinamento], [amostras[i] for i in reservadas])
    cache["apoios_reserva"], cache["rotulos_reserva"] = a, r
    salvar_cache()
print("Previsões alinhadas e registradas. As regras ainda serão comparadas.")

Divisão 42, rede 7: 0.2 s
Divisão 73, rede 7: 0.2 s
Divisão 101, rede 7: 0.2 s
Divisão 21, rede 7: 0.2 s
Divisão 145, rede 7: 0.2 s
Divisão 307, rede 7: 0.2 s
Divisão 509, rede 7: 0.2 s
Divisão 911, rede 7: 0.2 s
Divisão 42, rede 19: 0.2 s
Divisão 42, rede 31: 0.2 s
Previsões alinhadas e registradas. As regras ainda serão comparadas.


### Quais regras entram na busca

Definimos uma grade finita e percorremos todas as suas configurações. Contamos configurações de regras; regras diferentes podem produzir previsões iguais nesta base.

| Família | O que comparamos |
|---|---|
| Média dos apoios | Pesos de 0% a 100%, em passos de 2,5%, somando 100%; limite de 30% a 70%, em passos de 0,5%; incluímos a média exata de um terço para cada modelo |
| Votação | Todas as regras de votação monotônicas não constantes de três votos, incluindo maioria, concordância de pares e exigência de todos; limites individuais de 30% a 70%, em passos de 5% |
| Menor, maior e mediana dos apoios | Cada par e o trio, com os mesmos limites da média |
| Discordância entre dois modelos | Um modelo como referência, outro podendo substituí-lo conforme a diferença dos apoios e os limites exigidos |
| Discordância entre três modelos | Maioria dos votos, com possibilidade de usar o modelo de apoio mais forte quando a vantagem atinge o limite comparado |
| Disponibilidade dos exames | Regras diferentes conforme glicose ou hemoglobina estejam ausentes; comparamos os modelos individuais e as médias simples dos pares e do trio |

Uma votação monotônica mantém a decisão ou aumenta o voto para diabetes quando um componente muda de controle para diabetes. Evitamos regras em que mais votos para diabetes, sozinhos, forcem a saída para controle.

Em uma média exatamente no limite, usamos a classe individual do componente de maior peso; em empate de peso, a ordem é KNN, árvore e rede. Um modelo de peso zero não participa do desempate. Nas regras de votação, um apoio exatamente no limite mantém o voto individual daquele modelo.

In [10]:
nomes_modelos = ("KNN", "Árvore", "Rede")
regras = []


def registrar(familia, ativos, **parametros):
    regras.append({"familia": familia, "ativos": sorted(ativos), **parametros})


limites = [float(round(0.30 + i * 0.005, 6)) for i in range(81)]
pesos_medias = []
for a in range(41):
    for b in range(41 - a):
        pesos_medias.append([a / 40, b / 40, (40 - a - b) / 40])
pesos_medias.append([1 / 3] * 3)
for pesos in pesos_medias:
    ativos = [j for j, peso in enumerate(pesos) if peso > 0]
    for limite in limites:
        registrar("media", ativos, pesos=pesos, limite=limite)

limites_votos = [float(round(0.30 + i * 0.05, 6)) for i in range(9)]
mapas_monotonicos = []
for mapa in range(1, 255):
    respostas = [(mapa >> i) & 1 for i in range(8)]
    monotonico = all(respostas[i] <= respostas[i | (1 << j)] for i in range(8) for j in range(3))
    if not monotonico:
        continue
    ativos = [j for j in range(3) if any(respostas[i] != respostas[i ^ (1 << j)] for i in range(8))]
    mapas_monotonicos.append(mapa)
    for cortes in product(limites_votos, repeat=len(ativos)):
        por_modelo = [0.5] * 3
        for j, corte in zip(ativos, cortes):
            por_modelo[j] = corte
        registrar("votacao", ativos, mapa=mapa, limites=por_modelo)

for tamanho in (2, 3):
    for ativos in combinations(range(3), tamanho):
        for operacao in ("minimo", "maximo", "mediana"):
            for limite in limites:
                registrar("agregacao", ativos, operacao=operacao, limite=limite)

for principal in range(3):
    for auxiliar in range(3):
        if principal == auxiliar:
            continue
        for maximo_principal, minimo_auxiliar, vantagem in product(
            (0.55, 0.65, 0.75, 0.85, 1.0), (0.65, 0.75, 0.85, 0.95, 1.0), (0.0, 0.05, 0.10, 0.20)
        ):
            registrar("discordancia_par", (principal, auxiliar), principal=principal, auxiliar=auxiliar,
                      maximo_principal=maximo_principal, minimo_auxiliar=minimo_auxiliar, vantagem=vantagem)

for pesos, vantagem, minimo in product(product((0.5, 1.0, 2.0), repeat=3), (0.0, 0.025, 0.05, 0.1, 0.15, 0.2), (0.6, 0.7, 0.8, 0.9)):
    registrar("discordancia_trio", (0, 1, 2), pesos=list(pesos), vantagem=vantagem, minimo=minimo)

bases = [[1., 0., 0.], [0., 1., 0.], [0., 0., 1.], [.5, .5, 0.], [.5, 0., .5], [0., .5, .5], [1/3]*3]
for condicao, base, alternativa in product(("glicose_ausente", "hemoglobina_ausente", "algum_ausente", "ambos_ausentes", "ambos_presentes"), bases, bases):
    if base == alternativa:
        continue
    ativos = [j for j in range(3) if base[j] > 0 or alternativa[j] > 0]
    registrar("ausencias", ativos, condicao=condicao, base=base, alternativa=alternativa)

print("Configurações:", len(regras))
for familia in sorted({r["familia"] for r in regras}):
    print(familia, sum(r["familia"] == familia for r in regras))
print("Mapas de votação:", len(mapas_monotonicos))

Configurações: 79326
agregacao 972
ausencias 210
discordancia_par 600
discordancia_trio 648
media 69822
votacao 7074
Mapas de votação: 18


#### Como aplicamos uma regra

As funções abaixo recebem apenas os apoios, os votos individuais e a informação de ausência dos exames. Elas não recebem a classe verdadeira. Essa classe é usada depois, para contar acertos e erros.

Nas regras de discordância, chamamos de força do apoio a distância até 0,5. Por exemplo, 0,9 está mais distante de 0,5 que 0,6. Isso é uma medida operacional para esta comparação; não prova que um modelo seja mais confiável que outro.

In [11]:
def decidir(apoio, limite, desempate):
    previstas = (apoio > limite).astype(np.int8)
    empates = np.isclose(apoio, limite, rtol=0, atol=1e-12)
    previstas[empates] = desempate[empates]
    return previstas


def media_pesada(pesos, limite, apoios, rotulos):
    pesos = np.asarray(pesos)
    principal = int(np.argmax(pesos))
    return decidir(apoios @ pesos, limite, rotulos[:, principal])


def aplicar_regra(regra, apoios, rotulos, faltas):
    familia = regra["familia"]
    if familia == "media":
        return media_pesada(regra["pesos"], regra["limite"], apoios, rotulos)
    if familia == "votacao":
        limites = np.array(regra["limites"])
        votos = apoios > limites
        empates = np.isclose(apoios, limites, rtol=0, atol=1e-12)
        votos[empates] = rotulos[empates]
        codigo = votos.astype(np.int16) @ np.array([1, 2, 4], dtype=np.int16)
        return ((regra["mapa"] >> codigo) & 1).astype(np.int8)
    if familia == "agregacao":
        a = apoios[:, regra["ativos"]]
        operacao = {"minimo": np.min, "maximo": np.max, "mediana": np.median}[regra["operacao"]]
        return decidir(operacao(a, axis=1), regra["limite"], rotulos[:, regra["ativos"][0]])
    if familia == "discordancia_par":
        p, s = regra["principal"], regra["auxiliar"]
        mp, ms = np.abs(apoios[:, p] - 0.5), np.abs(apoios[:, s] - 0.5)
        trocar = ((rotulos[:, p] != rotulos[:, s]) & (ms - mp >= regra["vantagem"] - 1e-12)
                  & (mp <= regra["maximo_principal"] - 0.5 + 1e-12)
                  & (ms >= regra["minimo_auxiliar"] - 0.5 - 1e-12))
        return np.where(trocar, rotulos[:, s], rotulos[:, p]).astype(np.int8)
    if familia == "discordancia_trio":
        margens = np.abs(apoios - 0.5)
        forcas = margens * np.array(regra["pesos"])
        vencedor = np.argmax(forcas, axis=1)
        ordenadas = np.sort(forcas, axis=1)
        linhas = np.arange(len(apoios))
        usar = ((ordenadas[:, -1] - ordenadas[:, -2] >= regra["vantagem"] - 1e-12)
                & (margens[linhas, vencedor] >= regra["minimo"] - 0.5 - 1e-12))
        maioria = (rotulos.sum(axis=1) >= 2).astype(np.int8)
        return np.where(usar, rotulos[linhas, vencedor], maioria).astype(np.int8)
    if familia == "ausencias":
        g, h = faltas[:, 0], faltas[:, 1]
        mascara = {"glicose_ausente": g, "hemoglobina_ausente": h, "algum_ausente": g | h,
                   "ambos_ausentes": g & h, "ambos_presentes": ~(g | h)}[regra["condicao"]]
        base = media_pesada(regra["base"], 0.5, apoios, rotulos)
        alternativa = media_pesada(regra["alternativa"], 0.5, apoios, rotulos)
        return np.where(mascara, alternativa, base).astype(np.int8)
    raise ValueError(f"Família desconhecida: {familia}")


def ausencias(amostras_entrada):
    return np.array([[ler_numero(a, c) is None for c in ("GLUCOSE", "HEMOGLOBIN")] for a in amostras_entrada])


faltas_treinamento = ausencias([amostras[i] for i in treinamento])
faltas_reserva = ausencias([amostras[i] for i in reservadas])
faltas_teste = ausencias(teste)
reais_treinamento = classes[treinamento]


def avaliar_lista(lista, apoios, rotulos, partes, faltas, reais):
    contagens = np.zeros((5, len(lista), 4), dtype=np.int16)
    for inicio in range(0, len(lista), 512):
        fim = min(inicio + 512, len(lista))
        previstas = np.column_stack([aplicar_regra(r, apoios, rotulos, faltas) for r in lista[inicio:fim]])
        for parte in range(5):
            mascara = partes == parte
            p = previstas[mascara] == 1
            y = reais[mascara, None] == 1
            contagens[parte, inicio:fim] = np.column_stack((
                (~p & ~y).sum(axis=0), (p & ~y).sum(axis=0),
                (~p & y).sum(axis=0), (p & y).sum(axis=0),
            ))
    return contagens


prioridade_familia = {"media": 0, "votacao": 1, "agregacao": 2, "ausencias": 3,
                     "discordancia_par": 4, "discordancia_trio": 5}


def chave_ordem(indice, f1, lista):
    r = lista[indice]
    return (-float(f1[indice]), len(r["ativos"]), prioridade_familia[r["familia"]],
            abs(r.get("limite", 0.5) - 0.5), len(json.dumps(r)), indice)


def nome_regra(regra):
    ativos = "+".join(nomes_modelos[j] for j in regra["ativos"])
    if regra["familia"] == "media":
        pesos = ", ".join(f"{nomes_modelos[j]} {peso:.1%}" for j, peso in enumerate(regra["pesos"]) if peso > 0)
        return f"Média: {pesos}; limite {regra['limite']:.1%}"
    if regra["familia"] == "votacao":
        termos = []
        for padrao in range(8):
            if not ((regra["mapa"] >> padrao) & 1):
                continue
            if any((regra["mapa"] >> (padrao ^ (1 << j))) & 1 for j in range(3) if padrao & (1 << j)):
                continue
            termos.append(" e ".join(f"{nomes_modelos[j]}>{regra['limites'][j]:.0%}" for j in range(3) if padrao & (1 << j)))
        return "Votação: " + " OU ".join(termos)
    return f"{regra['familia']} ({ativos}): " + json.dumps({k: v for k, v in regra.items() if k not in {"familia", "ativos"}}, ensure_ascii=False)


def indice_media(pesos, limite=0.5):
    return next(i for i, r in enumerate(regras) if r["familia"] == "media"
                and np.allclose(r["pesos"], pesos, rtol=0, atol=1e-12)
                and abs(r["limite"] - limite) < 1e-12)


indice_original = indice_media([0.5, 0.5, 0.0])
indice_uniforme = indice_media([1/3] * 3)
assert np.array_equal(aplicar_regra(regras[indice_original], apoios_teste, rotulos_teste, faltas_teste),
                      combinar(apoios_teste[:, 0], apoios_teste[:, 1], 0.5))

### Busca ampla e primeira seleção

Comparamos todas as regras nas três primeiras sementes. Guardamos as matrizes por parte, e não somente a média final. Conferimos também se as respostas dos componentes coincidem com os registros anteriores nas divisões já usadas.

A lista seguinte terá até 12 regras de destaque no geral, oito de pares e oito do trio com previsões diferentes nas três divisões. Acrescentamos a melhor de cada família, os pares de pesos iguais, o trio de pesos iguais, os três modelos individuais e a mistura original. A segunda seleção vai comparar essa lista nas três sementes adicionais.

In [12]:
with np.load(raiz / "resultados" / "knn11_arvore.npz", allow_pickle=False) as anterior:
    for numero_semente, semente in enumerate((42, 73)):
        a, r, partes = obter_divisao(semente)
        for parte in range(5):
            mascara = partes == parte
            esperado = anterior["rodadas"][numero_semente * 5 + parte]
            assert np.array_equal(contar(reais_treinamento[mascara], r[mascara, 1]), esperado[1])
            previsto = aplicar_regra(regras[indice_original], a, r, faltas_treinamento)
with np.load(raiz / "resultados" / "rede_neural_exemplos_busca.npz", allow_pickle=False) as anterior:
    _, r, partes = obter_divisao(42)
    for parte in range(5):
        mascara = partes == parte
        assert np.array_equal(contar(reais_treinamento[mascara], r[mascara, 2]), anterior["busca"][parte, int(anterior["escolhida"])])
print("Árvore e rede mantêm as contagens locais anteriores; o KNN usa a nova escala.")

inicio = perf_counter()
busca_ampla = []
for semente in sementes_busca:
    a, r, partes = obter_divisao(semente)
    busca_ampla.append(avaliar_lista(regras, a, r, partes, faltas_treinamento, reais_treinamento))
    print(f"Todas as regras comparadas na semente {semente}", flush=True)
busca_ampla = np.concatenate(busca_ampla, axis=0)
f1_busca = metricas(busca_ampla)["f1"].mean(axis=0)
ordem_busca = sorted(range(len(regras)), key=lambda i: chave_ordem(i, f1_busca, regras))
elegiveis = [i for i in ordem_busca if len(regras[i]["ativos"]) >= 2]
lista_indices = []
assinaturas = set()


def assinatura(indice):
    pedacos = []
    for semente in sementes_busca:
        a, r, _ = obter_divisao(semente)
        pedacos.append(aplicar_regra(regras[indice], a, r, faltas_treinamento))
    return np.packbits(np.concatenate(pedacos)).tobytes()


for filtro, quantidade in ((None, 12), (2, 8), (3, 8)):
    adicionados = 0
    for i in elegiveis:
        if filtro is not None and len(regras[i]["ativos"]) != filtro:
            continue
        chave = assinatura(i)
        if chave in assinaturas:
            continue
        assinaturas.add(chave)
        lista_indices.append(i)
        adicionados += 1
        if adicionados == quantidade:
            break
for familia in prioridade_familia:
    i = next(i for i in elegiveis if regras[i]["familia"] == familia)
    if i not in lista_indices:
        lista_indices.append(i)
for pesos in bases:
    i = indice_media(pesos)
    if i not in lista_indices:
        lista_indices.append(i)
if indice_original not in lista_indices:
    lista_indices.append(indice_original)
lista_regras = [regras[i] for i in lista_indices]
print(f"Tempo da busca de regras: {perf_counter() - inicio:.1f} s")
print("Regras na segunda seleção:", len(lista_indices))
for i in elegiveis[:10]:
    print(f"F1 {f1_busca[i]:.2%} | {nome_regra(regras[i])}")

caminho_busca = raiz / "resultados" / "combinacoes_zscore_busca.npz"
np.savez_compressed(caminho_busca,
    catalogo=json.dumps({"regras": regras, "sementes_busca": sementes_busca,
                        "lista_indices": lista_indices, "sha_base": sha_base, "versao": versao_modelos}, ensure_ascii=False),
    busca=busca_ampla)

Árvore e rede mantêm as contagens locais anteriores; o KNN usa a nova escala.
Todas as regras comparadas na semente 42
Todas as regras comparadas na semente 73
Todas as regras comparadas na semente 101
Tempo da busca de regras: 11.4 s
Regras na segunda seleção: 39
F1 91.34% | Média: KNN 40.0%, Árvore 32.5%, Rede 27.5%; limite 46.5%
F1 91.33% | Média: KNN 35.0%, Árvore 37.5%, Rede 27.5%; limite 44.5%
F1 91.26% | Média: KNN 40.0%, Árvore 37.5%, Rede 22.5%; limite 46.5%
F1 91.25% | Média: KNN 40.0%, Árvore 32.5%, Rede 27.5%; limite 45.5%
F1 91.25% | Média: KNN 32.5%, Árvore 42.5%, Rede 25.0%; limite 45.5%
F1 91.25% | Média: KNN 40.0%, Árvore 32.5%, Rede 27.5%; limite 46.0%
F1 91.25% | Média: KNN 37.5%, Árvore 37.5%, Rede 25.0%; limite 45.5%
F1 91.25% | Média: KNN 32.5%, Árvore 37.5%, Rede 30.0%; limite 44.5%
F1 91.24% | Média: KNN 37.5%, Árvore 37.5%, Rede 25.0%; limite 42.5%
F1 91.24% | Média: KNN 37.5%, Árvore 37.5%, Rede 25.0%; limite 43.0%


### Segunda seleção e escolha fixada

As melhores regras da busca ampla podem ter aproveitado particularidades das primeiras divisões. Agora a lista menor recebe resultados nas sementes 21, 145 e 307. Escolhemos pela média do F1 macro das 30 partes das seis sementes.

Registramos a melhor regra com dois componentes e a melhor com três. A indicação principal é a de maior F1 macro médio nessa seleção. As conferências seguintes e a reserva não trocam essa indicação.

In [13]:
refinamento = []
for semente in sementes_refinamento:
    a, r, partes = obter_divisao(semente)
    refinamento.append(avaliar_lista(lista_regras, a, r, partes, faltas_treinamento, reais_treinamento))
refinamento = np.concatenate(refinamento, axis=0)
selecao = np.concatenate((busca_ampla[:, lista_indices], refinamento), axis=0)
f1_selecao = metricas(selecao)["f1"].mean(axis=0)
ordem_selecao = sorted(range(len(lista_regras)), key=lambda i: chave_ordem(i, f1_selecao, lista_regras))
candidatas = [i for i in ordem_selecao if len(lista_regras[i]["ativos"]) >= 2]
melhor_dupla = next(i for i in candidatas if len(lista_regras[i]["ativos"]) == 2)
melhor_trio = next(i for i in candidatas if len(lista_regras[i]["ativos"]) == 3)
escolhida = candidatas[0]
original_na_lista = lista_indices.index(indice_original)
uniforme_na_lista = lista_indices.index(indice_uniforme)
print("Indicação fixada:", nome_regra(lista_regras[escolhida]))
print("Melhor dupla:", nome_regra(lista_regras[melhor_dupla]))
print("Melhor trio:", nome_regra(lista_regras[melhor_trio]))
print("Avaliação                 | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
for nome, i in (("KNN+árvore, pesos iguais", original_na_lista), ("Trio, pesos iguais", uniforme_na_lista),
                ("Melhor dupla", melhor_dupla), ("Melhor trio", melhor_trio), ("Indicação principal", escolhida)):
    mostrar(nome, selecao[:, i])
    print("F1 por semente:", np.round(metricas(selecao[:, i])["f1"].reshape(6, 5).mean(axis=1), 4).tolist())

escolha = {"regra": lista_regras[escolhida], "indice_global": lista_indices[escolhida],
           "melhor_dupla": lista_regras[melhor_dupla], "melhor_trio": lista_regras[melhor_trio],
           "f1_selecao": float(f1_selecao[escolhida]), "sha_base": sha_base,
           "versao_modelos": versao_modelos, "sementes_selecao": list(sementes_busca + sementes_refinamento),
           "criterio": "maior F1 macro médio nas 30 partes; conferências e reserva não escolhem a regra"}
caminho_escolha = raiz / "resultados" / "combinacoes_zscore_escolha.json"
caminho_escolha.write_text(json.dumps(escolha, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
np.savez_compressed(raiz / "resultados" / "combinacoes_zscore_selecao.npz", selecao=selecao,
                    lista_indices=lista_indices, escolhida=escolhida, melhor_dupla=melhor_dupla,
                    melhor_trio=melhor_trio, original_na_lista=original_na_lista, uniforme_na_lista=uniforme_na_lista)

Indicação fixada: Média: KNN 32.5%, Árvore 42.5%, Rede 25.0%; limite 45.5%
Melhor dupla: Média: KNN 52.5%, Árvore 47.5%; limite 47.0%
Melhor trio: Média: KNN 32.5%, Árvore 42.5%, Rede 25.0%; limite 45.5%
Avaliação                 | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
KNN+árvore, pesos iguais |   90.67% |   90.65% |   90.91% |   91.50% |   89.91%
F1 por semente: [0.906, 0.9158, 0.8895, 0.906, 0.9158, 0.9059]
Trio, pesos iguais       |   90.99% |   90.98% |   91.13% |   92.25% |   89.82%
F1 por semente: [0.9086, 0.9182, 0.8943, 0.9159, 0.9133, 0.9083]
Melhor dupla             |   90.91% |   90.88% |   91.10% |   90.33% |   91.45%
F1 por semente: [0.9133, 0.9157, 0.8967, 0.9082, 0.906, 0.9132]
Melhor trio              |   91.55% |   91.53% |   91.72% |   90.75% |   92.30%
F1 por semente: [0.9181, 0.9229, 0.8966, 0.9205, 0.9156, 0.918]
Indicação principal      |   91.55% |   91.53% |   91.72% |   90.75% |   92.30%
F1 por semente: [0.9181, 0.9229, 0.8966, 0.9205, 0.9156, 0.

### Conferências após a escolha

Mantemos as regras fixadas e observamos duas divisões adicionais, 509 e 911. Também conferimos a divisão 42 com as inicializações 19 e 31 da rede. A tabela permite acompanhar o efeito de mudar os exemplos de cada parte ou o começo do aprendizado da rede.

Essas conferências continuam usando as 416 pessoas de treinamento. A variação entre divisões ajuda a avaliar a estabilidade; ela não fornece uma garantia para o conjunto do Kaggle.

In [14]:
avaliadas = [("KNN+árvore, pesos iguais", original_na_lista), ("Trio, pesos iguais", uniforme_na_lista),
             ("Melhor dupla", melhor_dupla), ("Melhor trio", melhor_trio), ("Indicação principal", escolhida)]
conferencias = []
nomes_conferencias = []
for divisao, semente_rede in [(509, 7), (911, 7), (42, 19), (42, 31)]:
    a, r, partes = obter_divisao(divisao, semente_rede)
    contagens = avaliar_lista([lista_regras[i] for _, i in avaliadas], a, r, partes, faltas_treinamento, reais_treinamento)
    conferencias.append(contagens)
    nomes_conferencias.append((divisao, semente_rede))
    print(f"Divisão {divisao}, inicialização da rede {semente_rede}")
    for j, (nome, _) in enumerate(avaliadas):
        mostrar(nome, contagens[:, j])
conferencias = np.array(conferencias)

Divisão 509, inicialização da rede 7
KNN+árvore, pesos iguais |   89.43% |   89.39% |   89.79% |   90.00% |   88.87%
Trio, pesos iguais       |   90.39% |   90.34% |   90.78% |   90.50% |   90.26%
Melhor dupla             |   89.43% |   89.36% |   89.94% |   87.50% |   91.18%
Melhor trio              |   90.63% |   90.56% |   91.26% |   88.50% |   92.58%
Indicação principal      |   90.63% |   90.56% |   91.26% |   88.50% |   92.58%
Divisão 911, inicialização da rede 7
KNN+árvore, pesos iguais |   88.94% |   88.92% |   89.00% |   89.50% |   88.41%
Trio, pesos iguais       |   90.62% |   90.61% |   90.72% |   91.00% |   90.27%
Melhor dupla             |   89.90% |   89.88% |   89.92% |   89.00% |   90.73%
Melhor trio              |   91.58% |   91.56% |   91.70% |   90.00% |   93.03%
Indicação principal      |   91.58% |   91.56% |   91.70% |   90.00% |   93.03%
Divisão 42, inicialização da rede 19
KNN+árvore, pesos iguais |   90.63% |   90.60% |   90.89% |   91.50% |   89.80%
Trio, pes

### Comparação nas 104 amostras reservadas

Agora observamos a reserva, mantendo as escolhas anteriores. Comparamos os três modelos individuais, a mistura já enviada e as regras selecionadas. A matriz usa linhas para a classe verdadeira e colunas para a prevista, na ordem controle e diabetes.

Registramos também quantos erros da mistura original foram corrigidos e quantos acertos ela perdeu. A reserva já foi consultada em outros experimentos do projeto e não será usada para escolher novos pesos nesta etapa.

In [15]:
a_reserva, r_reserva = cache["apoios_reserva"], cache["rotulos_reserva"]
y_reserva = classes[reservadas]
previstas_reserva = [aplicar_regra(lista_regras[i], a_reserva, r_reserva, faltas_reserva) for _, i in avaliadas]
contagens_reserva = np.array([contar(y_reserva, p) for p in previstas_reserva])
print("Avaliação                 | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
for j, nome in enumerate(nomes_modelos):
    mostrar(nome, contar(y_reserva, r_reserva[:, j]))
for j, (nome, _) in enumerate(avaliadas):
    mostrar(nome, contagens_reserva[j])
    print("Matriz:", contagens_reserva[j].reshape(2, 2).tolist())
original, principal = previstas_reserva[0], previstas_reserva[-1]
print("Erros da média 50/50 corrigidos:", int(np.sum((original != y_reserva) & (principal == y_reserva))))
print("Acertos da média 50/50 perdidos:", int(np.sum((original == y_reserva) & (principal != y_reserva))))
for j, k in combinations(range(3), 2):
    print(f"Discordâncias {nomes_modelos[j]} / {nomes_modelos[k]}:", int(np.sum(r_reserva[:, j] != r_reserva[:, k])))
assert np.array_equal(contar(y_reserva, r_reserva[:, 1]), [43, 7, 9, 45])
assert np.array_equal(contar(y_reserva, r_reserva[:, 2]), [44, 6, 9, 45])
np.savez_compressed(raiz / "resultados" / "combinacoes_zscore_conferencias.npz", conferencias=conferencias,
                    nomes_conferencias=nomes_conferencias, reserva=contagens_reserva,
                    avaliadas=json.dumps(avaliadas, ensure_ascii=False))
mostrar("Mistura anterior com IQR", np.array([44, 6, 7, 47]))

Avaliação                 | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
KNN                      |   75.96% |   75.94% |   75.93% |   76.00% |   75.93%
Árvore                   |   84.62% |   84.61% |   84.62% |   86.00% |   83.33%
Rede                     |   85.58% |   85.58% |   85.63% |   88.00% |   83.33%
KNN+árvore, pesos iguais |   85.58% |   85.56% |   85.55% |   86.00% |   85.19%
Matriz: [[43, 7], [8, 46]]
Trio, pesos iguais       |   86.54% |   86.53% |   86.54% |   88.00% |   85.19%
Matriz: [[44, 6], [8, 46]]
Melhor dupla             |   84.62% |   84.56% |   84.67% |   82.00% |   87.04%
Matriz: [[41, 9], [7, 47]]
Melhor trio              |   86.54% |   86.49% |   86.61% |   84.00% |   88.89%
Matriz: [[42, 8], [6, 48]]
Indicação principal      |   86.54% |   86.49% |   86.61% |   84.00% |   88.89%
Matriz: [[42, 8], [6, 48]]
Erros da média 50/50 corrigidos: 2
Acertos da média 50/50 perdidos: 1
Discordâncias KNN / Árvore: 23
Discordâncias KNN / Rede: 16
Discordâncias 

### Arquivos para o Kaggle

Aplicamos a regra fixada aos componentes ajustados nas 520 amostras. Geramos a indicação principal e deixamos os melhores pares e trio como alternativas identificadas. A recomendação principal segue o critério registrado antes das conferências.

Os arquivos têm 312 previsões, duas colunas e os IDs do teste na ordem original. As previsões do teste não participam da escolha das regras. O resultado no Kaggle só será conhecido após o envio.

In [16]:
saidas = [("submissao_combinacao_zscore.csv", escolhida),
          ("submissao_dupla_zscore.csv", melhor_dupla), ("submissao_trio_zscore.csv", melhor_trio)]
previsoes_salvas = {}
for nome, i in saidas:
    previstas = aplicar_regra(lista_regras[i], apoios_teste, rotulos_teste, faltas_teste)
    caminho = raiz / "resultados" / nome
    with caminho.open("w", encoding="utf-8", newline="") as arquivo:
        escritor = csv.writer(arquivo)
        escritor.writerow(["id", "class"])
        escritor.writerows(zip(ids_teste, previstas.tolist(), strict=True))
    with caminho.open(encoding="utf-8", newline="") as arquivo:
        leitor = csv.DictReader(arquivo)
        assert leitor.fieldnames == ["id", "class"]
        linhas = list(leitor)
    assert len(linhas) == 312
    assert [a["id"] for a in linhas] == ids_teste
    assert all(a["class"] in {"0", "1"} for a in linhas)
    assert [int(a["class"]) for a in linhas] == previstas.tolist()
    previsoes_salvas[nome] = previstas
    print(nome, "|", nome_regra(lista_regras[i]))
    print("SHA-256:", hashlib.sha256(caminho.read_bytes()).hexdigest())

carregada = json.loads(caminho_escolha.read_text(encoding="utf-8"))
reaberta = aplicar_regra(carregada["regra"], cache["apoios_teste"], cache["rotulos_teste"], faltas_teste)
assert np.array_equal(reaberta, previsoes_salvas[saidas[0][0]])
print("Regra reaberta reproduz as 312 previsões da indicação principal.")
print("Diferenças em relação à média 50/50 desta escala:", int(np.sum(reaberta != aplicar_regra(regras[indice_original], apoios_teste, rotulos_teste, faltas_teste))))

caminho_resumo = raiz / "resultados" / "combinacoes_zscore_resumo.csv"
with caminho_resumo.open("w", encoding="utf-8", newline="") as arquivo:
    escritor = csv.writer(arquivo)
    escritor.writerow(["regra", "componentes", "descricao", "f1_selecao", "f1_conferencia_509", "f1_conferencia_911", "f1_pesos_19", "f1_pesos_31", "f1_reserva"])
    for j, (nome, i) in enumerate(avaliadas):
        escritor.writerow([nome, len(lista_regras[i]["ativos"]), nome_regra(lista_regras[i]), float(f1_selecao[i]),
                          *[float(metricas(conferencias[k, :, j])["f1"].mean()) for k in range(4)],
                          float(metricas(contagens_reserva[j])["f1"])])

submissao_combinacao_zscore.csv | Média: KNN 32.5%, Árvore 42.5%, Rede 25.0%; limite 45.5%
SHA-256: 5eac2683ae79c2e74e2b94c9143366d98adec375418180ea09fd54066688c6db
submissao_dupla_zscore.csv | Média: KNN 52.5%, Árvore 47.5%; limite 47.0%
SHA-256: 7bf5ce52532fc1470421b6d45ad1dd2b14edff230f31ea312d33d8530b8c6e20
submissao_trio_zscore.csv | Média: KNN 32.5%, Árvore 42.5%, Rede 25.0%; limite 45.5%
SHA-256: 5eac2683ae79c2e74e2b94c9143366d98adec375418180ea09fd54066688c6db
Regra reaberta reproduz as 312 previsões da indicação principal.
Diferenças em relação à média 50/50 desta escala: 16


### Como interpretar a comparação

A busca percorreu toda a grade declarada, com componentes fixos. Outras formas de misturar modelos e outros valores de parâmetros ainda existem. O melhor resultado aqui é o melhor segundo o critério local registrado, não uma promessa de maior pontuação pública.

Testar muitas regras pode favorecer coincidências das divisões usadas para escolher. Por isso separamos a seleção, as conferências e a observação da reserva, e preservamos os resultados completos. As combinações que usam somente dois modelos continuam válidas, mesmo após testar o trio.

### Referências e origem dos componentes

- [Mistura original](../Mod5_KNN_AD/01_Knn_arvore.ipynb): KNN K=11 e árvore de quatro atributos.
- [Rede neural](../Mod3_RN/02_Rede_neural_exemplo_a_exemplo.ipynb): atualização após cada exemplo, quatro entradas, 12 neurônios ocultos e uma saída.
- [Escopo das aulas](../docs/ESCOPO_AULAS.md): conceitos apresentados e escolhas da implementação.
- [Aula 5](../docs/materiais_aula/aula5_MD.pdf): KNN e árvores.
- [Aula 6](../docs/materiais_aula/aula6_MD.pdf): redes neurais.
- [Aula 7](../docs/materiais_aula/aula7_MD.pdf): avaliação dos classificadores.